# Fine-tune Llama 3.2 3B on FineTome-100k with Unsloth (LoRA, 4-bit)

Needs a CUDA GPU. A free Colab T4 works: Runtime -> Change runtime type -> T4 GPU.
Same steps as the local run in `src/finetome_sft/`: ShareGPT -> chat messages ->
chat template -> loss on assistant tokens only -> LoRA SFT -> held-out loss.
This file doubles as the notebook source (`# %%` cells); see the .ipynb next to it.

In [ ]:
!pip install -q unsloth

In [ ]:
import math

import torch
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

MODEL = "unsloth/Llama-3.2-3B-Instruct"
MAX_LEN = 2048
N_HELDOUT = 500
MAX_STEPS = 60

## 1. Load the model in 4-bit and add LoRA adapters

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_LEN, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 2. ShareGPT -> chat messages -> Llama 3.1 chat template text

In [ ]:
ROLE_MAP = {"human": "user", "user": "user", "gpt": "assistant", "assistant": "assistant", "system": "system"}


def sharegpt_to_messages(conversations):
    return [{"role": ROLE_MAP[t["from"].lower()], "content": t["value"].strip()} for t in conversations]


tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")


def to_text(batch):
    texts = [tokenizer.apply_chat_template(sharegpt_to_messages(c), tokenize=False) for c in batch["conversations"]]
    return {"text": texts}


ds = load_dataset("mlabonne/FineTome-100k", split="train").shuffle(seed=3407)
ds = ds.map(to_text, batched=True, remove_columns=ds.column_names)
split = ds.train_test_split(test_size=N_HELDOUT, seed=3407)
train_ds, held_ds = split["train"], split["test"]
print(train_ds[0]["text"][:600])

## 3. Train on assistant tokens only

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=held_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs/llama-3.2-3b",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

before = trainer.evaluate()["eval_loss"]
stats = trainer.train()
after = trainer.evaluate()["eval_loss"]
print(f"held-out loss {before:.4f} -> {after:.4f}  (ppl {math.exp(before):.2f} -> {math.exp(after):.2f})")

## 4. Try it, then save the adapter (a few MB, not the 3B weights)

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "Give exactly three tips for keeping houseplants alive. Use a bulleted list with '-' and nothing else."}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True))

model.save_pretrained("outputs/llama-3.2-3b/adapter")
tokenizer.save_pretrained("outputs/llama-3.2-3b/adapter")